In [1]:

from sklearn.datasets import load_wine
import pandas as pd

wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

print("Dataset: Wine (178 samples, 13 features, 3 classes)")
print("Shape:", df.shape)
print(df.head())



Dataset: Wine (178 samples, 13 features, 3 classes)
Shape: (178, 14)
   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.

In [2]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features — essential for Logistic Regression convergence
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)
y_pred = log_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 1.0

Confusion Matrix:
 [[14  0  0]
 [ 0 14  0]
 [ 0  0  8]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        14
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00         8

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



In [3]:

import numpy as np

samples = pd.concat([
    df[df['target']==0].iloc[0:3],
    df[df['target']==1].iloc[0:2],
    df[df['target']==2].iloc[0:2]
]).reset_index(drop=True)

print("7 selected rows:")
print(samples)


7 selected rows:
   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    12.37        0.94  1.36               10.6       88.0           1.98   
4    12.33        1.10  2.28               16.0      101.0           2.05   
5    12.86        1.35  2.32               18.0      122.0           1.51   
6    12.88        2.99  2.40               20.0      104.0           1.30   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        0.57                  0.28             0.

In [4]:

new_sample = np.array([13.0, 2.0, 2.4, 16.0, 105.0, 2.5, 2.8, 0.30, 1.9, 5.0, 1.0, 3.2, 1000.0])
print("New Sample:", new_sample)

features_list = df.columns[:-1].tolist()
distances = []

for idx, row in samples.iterrows():
    point = row[features_list].values
    dist  = np.sqrt(np.sum((point - new_sample) ** 2))
    distances.append({'Row': idx, 'Class': int(row['target']), 'Distance': round(dist, 4)})

dist_df = pd.DataFrame(distances).sort_values('Distance').reset_index(drop=True)
print("\nEuclidean Distances (sorted):")
print(dist_df)


New Sample: [1.30e+01 2.00e+00 2.40e+00 1.60e+01 1.05e+02 2.50e+00 2.80e+00 3.00e-01
 1.90e+00 5.00e+00 1.00e+00 3.20e+00 1.00e+03]

Euclidean Distances (sorted):
   Row  Class  Distance
0    1      0   50.4879
1    0      0   68.6440
2    2      0  185.0664
3    4      1  320.0439
4    5      2  370.4083
5    6      2  470.0282
6    3      1  480.3534


In [5]:

for k in [3, 5, 7]:
    top_k           = dist_df.head(k)
    predicted_class = top_k['Class'].mode()[0]
    votes           = top_k['Class'].value_counts().to_dict()
    print(f"K={k}: Neighbors={top_k['Class'].tolist()}  Votes={votes}  → Predicted Class: {predicted_class}")


K=3: Neighbors=[0, 0, 0]  Votes={0: 3}  → Predicted Class: 0
K=5: Neighbors=[0, 0, 0, 1, 2]  Votes={0: 3, 1: 1, 2: 1}  → Predicted Class: 0
K=7: Neighbors=[0, 0, 0, 1, 2, 2, 1]  Votes={0: 3, 1: 2, 2: 2}  → Predicted Class: 0


In [6]:

from sklearn.datasets import load_iris

iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['target'] = iris.target

print(df_iris.shape)
print(df_iris.head())


(150, 5)
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  


In [7]:

samples_iris = pd.concat([
    df_iris[df_iris['target']==0].iloc[0:3],
    df_iris[df_iris['target']==1].iloc[0:2],
    df_iris[df_iris['target']==2].iloc[0:2]
])
print("7 selected rows:")
print(samples_iris)


7 selected rows:
     sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                  5.1               3.5                1.4               0.2   
1                  4.9               3.0                1.4               0.2   
2                  4.7               3.2                1.3               0.2   
50                 7.0               3.2                4.7               1.4   
51                 6.4               3.2                4.5               1.5   
100                6.3               3.3                6.0               2.5   
101                5.8               2.7                5.1               1.9   

     target  
0         0  
1         0  
2         0  
50        1  
51        1  
100       2  
101       2  


In [8]:

new_sample_iris = np.array([5.8, 3.0, 4.2, 1.3])
print("New sample:", new_sample_iris)


New sample: [5.8 3.  4.2 1.3]


In [9]:

distances_iris = []

for idx, row in samples_iris.iterrows():
    point = row[:-1].values
    dist  = np.sqrt(np.sum((point - new_sample_iris) ** 2))
    distances_iris.append({'index': idx, 'target': row['target'], 'distance': dist})

dist_df_iris = pd.DataFrame(distances_iris).sort_values('distance').reset_index(drop=True)
print(dist_df_iris)


   index  target  distance
0     51     1.0  0.728011
1    101     2.0  1.122497
2     50     1.0  1.319091
3    100     2.0  2.240536
4      0     0.0  3.128898
5      1     0.0  3.140064
6      2     0.0  3.296968


In [10]:

for k in [3, 5, 7]:
    top_k           = dist_df_iris.head(k)
    predicted_class = top_k['target'].mode()[0]
    print(f"K={k}:")
    print(top_k[['target', 'distance']])
    print(f"  → Predicted class: {predicted_class}")
    print(f"  → Votes: {top_k['target'].value_counts().to_dict()}")
    print()


K=3:
   target  distance
0     1.0  0.728011
1     2.0  1.122497
2     1.0  1.319091
  → Predicted class: 1.0
  → Votes: {1.0: 2, 2.0: 1}

K=5:
   target  distance
0     1.0  0.728011
1     2.0  1.122497
2     1.0  1.319091
3     2.0  2.240536
4     0.0  3.128898
  → Predicted class: 1.0
  → Votes: {1.0: 2, 2.0: 2, 0.0: 1}

K=7:
   target  distance
0     1.0  0.728011
1     2.0  1.122497
2     1.0  1.319091
3     2.0  2.240536
4     0.0  3.128898
5     0.0  3.140064
6     0.0  3.296968
  → Predicted class: 0.0
  → Votes: {0.0: 3, 1.0: 2, 2.0: 2}

